# 05. Latent World Models Rssm Dreamer Tdmpc and Rnd Curiosity

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Sutton (1990 Dyna); Burda et al. (2018 RND Curiosity); Ha & Schmidhuber (2018 World Models); Hafner et al. (2019 PlaNet, 2020/2023 DreamerV1–V3); Hansen et al. (2022/2024 TD-MPC2).
- **Purpose:** First-principles mathematical implementation and empirical verification of model-based reinforcement learning with latent state-space models: Recurrent State-Space Models (RSSM), KL Balancing, Random Network Distillation (RND) curiosity, and TD-MPC2 latent rollout evaluation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Offline RL Imitation Cql IQL Gail and Decision Transformer](04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. Hierarchical RL Options Feudal Her and POMDP Active Inference](06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never use symmetric unweighted KL divergence $D_{\text{KL}}(q_\phi \| p_\theta)$ when training Recurrent State-Space Models (RSSMs). Without **KL Balancing** ($\beta_{\text{prior}}=0.8, \beta_{\text{post}}=0.2$), the posterior encoder collapses toward the random untrained prior, ignoring sensory observations entirely. Do NOT use forward prediction error curiosity (predicting $s_{t+1}$ from $(s_t, a_t)$) in stochastic environments; the agent will become permanently fixated on unlearnable white noise (the *Noisy TV Problem*); use **RND** instead.

## Mathematical Foundations & Governing Equations

### 1. Gaussian Relative Entropy (KL Divergence)
$$
D_{\text{KL}}(q \| p) = \frac{1}{2} \sum_{i=1}^D \left( \frac{\sigma_{q,i}^2 + (\mu_{q,i} - \mu_{p,i})^2}{\sigma_{p,i}^2} - 1 + 2 (\log \sigma_{p,i} - \log \sigma_{q,i}) \right)
$$

### 2. RSSM KL Balancing
$$
\mathcal{L}_{\text{KL}} = \beta_{\text{prior}} \mathbb{E}\left[ D_{\text{KL}}(\text{sg}[q] \| p) \right] + \beta_{\text{post}} \mathbb{E}\left[ D_{\text{KL}}(q \| \text{sg}[p]) \right]
$$

### 3. Random Network Distillation (RND) Curiosity
$$
r_i(s) = \frac{1}{D} \sum_{d=1}^D (\hat{f}_\theta(s)_d - f_{\text{target}}(s)_d)^2
$$

### 4. TD-MPC2 Latent Imagination Return
$$
G(a_{0:H-1}) = \sum_{t=0}^{H-1} \gamma^t \hat{r}(z_t, a_t) + \gamma^H \hat{Q}(z_H, a_H)
$$

### Asymptotic Time & Memory Complexity
- Gaussian KL Divergence: $\mathcal{O}(B \cdot D)$ time and memory.
- RND Feature Distillation: $\mathcal{O}(B \cdot D)$ time and memory.
- TD-MPC2 Latent Rollout Return: $\mathcal{O}(B \cdot H)$ time and memory.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def gaussian_kl_divergence(
    mu_q: torch.Tensor,
    log_std_q: torch.Tensor,
    mu_p: torch.Tensor,
    log_std_p: torch.Tensor,
) -> torch.Tensor:
    var_q = torch.exp(2.0 * log_std_q)
    var_p = torch.exp(2.0 * log_std_p)
    diff = mu_q - mu_p
    kl_element = (var_q + diff ** 2) / var_p - 1.0 + 2.0 * (log_std_p - log_std_q)
    return 0.5 * torch.sum(kl_element, dim=-1)

def rssm_kl_balancing(
    mu_q: torch.Tensor,
    log_std_q: torch.Tensor,
    mu_p: torch.Tensor,
    log_std_p: torch.Tensor,
    beta_prior: float = 0.8,
    beta_post: float = 0.2,
) -> torch.Tensor:
    kl_prior = gaussian_kl_divergence(mu_q.detach(), log_std_q.detach(), mu_p, log_std_p)
    kl_post = gaussian_kl_divergence(mu_q, log_std_q, mu_p.detach(), log_std_p.detach())
    return beta_prior * torch.mean(kl_prior) + beta_post * torch.mean(kl_post)

def rnd_intrinsic_reward(
    target_features: torch.Tensor,
    predictor_features: torch.Tensor,
) -> torch.Tensor:
    return torch.mean((predictor_features - target_features) ** 2, dim=-1)

def tdmpc_latent_rollout_return(
    latent_rewards: torch.Tensor,
    terminal_q: torch.Tensor,
    gamma: float = 0.99,
) -> torch.Tensor:
    horizon = latent_rewards.shape[1]
    discounts = gamma ** torch.arange(horizon, dtype=torch.float32, device=latent_rewards.device)
    discounted_rewards = torch.sum(latent_rewards * discounts.unsqueeze(0), dim=-1)
    terminal_discount = gamma ** horizon
    return discounted_rewards + terminal_discount * terminal_q

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-6, f"Linear anchor parity error: {err_anchor:.2e}"

kl_div = gaussian_kl_divergence(weights["mu_q"], weights["log_std_q"], weights["mu_p"], weights["log_std_p"])
err_kl = float(torch.max(torch.abs(kl_div - weights["kl_div_ref"])))
assert err_kl < 1e-6, f"KL divergence parity error: {err_kl:.2e}"

kl_bal = rssm_kl_balancing(weights["mu_q"], weights["log_std_q"], weights["mu_p"], weights["log_std_p"], beta_prior=0.8, beta_post=0.2)
err_bal = float(torch.max(torch.abs(kl_bal - weights["kl_balanced_ref"])))
assert err_bal < 1e-6, f"KL balancing parity error: {err_bal:.2e}"

rnd_r = rnd_intrinsic_reward(weights["target_features"], weights["predictor_features"])
err_rnd = float(torch.max(torch.abs(rnd_r - weights["rnd_rewards_ref"])))
assert err_rnd < 1e-6, f"RND rewards parity error: {err_rnd:.2e}"

tdmpc_ret = tdmpc_latent_rollout_return(weights["latent_rewards"], weights["terminal_q"], gamma=0.99)
err_tdmpc = float(torch.max(torch.abs(tdmpc_ret - weights["tdmpc_return_ref"])))
assert err_tdmpc < 1e-6, f"TD-MPC return parity error: {err_tdmpc:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_kl, err_bal, err_rnd, err_tdmpc):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_kl = benchmark_fn(
    lambda: gaussian_kl_divergence(weights["mu_q"], weights["log_std_q"], weights["mu_p"], weights["log_std_p"]),
    iters=100,
    name="Gaussian KL Divergence",
)
b_rnd = benchmark_fn(
    lambda: rnd_intrinsic_reward(weights["target_features"], weights["predictor_features"]),
    iters=100,
    name="RND Curiosity Reward",
)
b_tdmpc = benchmark_fn(
    lambda: tdmpc_latent_rollout_return(weights["latent_rewards"], weights["terminal_q"]),
    iters=100,
    name="TD-MPC2 Latent Rollout",
)

print(format_benchmark_table([b_kl, b_rnd, b_tdmpc]))

In [ ]:
# Failure Mode Contrast: Posterior Collapse in Unbalanced KL vs KL Balancing
# Suppose the posterior encoder q has high-capacity expressive latents with informative signals (mu_q = 2.0),
# while the prior p is randomly initialized with zero-mean (mu_p = 0.0).
mu_q_test = torch.tensor([[2.0]], requires_grad=True)
log_std_q_test = torch.tensor([[0.0]], requires_grad=True)
mu_p_test = torch.tensor([[0.0]], requires_grad=True)
log_std_p_test = torch.tensor([[0.0]], requires_grad=True)

# 1. Unbalanced Symmetric KL: prior and posterior gradients have equal weight
loss_unbal = gaussian_kl_divergence(mu_q_test, log_std_q_test, mu_p_test, log_std_p_test).mean()
grad_q_unbal = torch.autograd.grad(loss_unbal, mu_q_test, retain_graph=True)[0].item()

# 2. KL Balanced: prior is trained 4x faster (beta_prior = 0.8), posterior regularized gently (beta_post = 0.2)
loss_bal = 0.8 * gaussian_kl_divergence(mu_q_test.detach(), log_std_q_test.detach(), mu_p_test, log_std_p_test).mean() + \
           0.2 * gaussian_kl_divergence(mu_q_test, log_std_q_test, mu_p_test.detach(), log_std_p_test.detach()).mean()
grad_q_bal = torch.autograd.grad(loss_bal, mu_q_test)[0].item()

print(f"Unbalanced Posterior Encoder Gradient: {grad_q_unbal:.4f} (Heavy gradient pulls encoder to collapse to prior!)")
print(f"KL Balanced Posterior Encoder Gradient:  {grad_q_bal:.4f} (Scaled down by 5x (0.2x), preventing collapse!)")

assert abs(grad_q_unbal - 2.0) < 1e-4, f"Unbalanced gradient must equal 2.0 (got {grad_q_unbal})"
assert abs(grad_q_bal - 0.4) < 1e-4, f"Balanced gradient on posterior must be 0.2 * 2.0 = 0.4 (got {grad_q_bal})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **KL Balancing**: Decoupling the training rates of the prior dynamic model ($\beta_{\text{prior}}=0.8$) and the posterior encoder ($\beta_{\text{post}}=0.2$) resolves posterior collapse without artificial capacity bottlenecks.
2. **Implicit World Models (TD-MPC2)**: Reconstructing raw pixels is unnecessary for control; learning compact latent spaces governed solely by reward and Q-value predictions reduces compute by $10\times$.
3. **Fixed Projection Curiosity (RND)**: Measuring error against static random targets prevents curiosity algorithms from falling into the "noisy TV" trap of chasing unlearnable stochastic entropy.

---
### Cross-Domain Generalisation Matrix
- **Robotics Locomotion**: DreamerV3 trains continuous quadruped gait control via 100,000 parallel imagined trajectories without robot wear-and-tear.
- **Autonomous Vehicles**: TD-MPC2 plans evasion trajectories across latent multi-agent vehicle interaction states at 50Hz on embedded hardware.
- **Algorithmic Maze Exploration**: RND intrinsic curiosity drives exploration in sparse-reward environments with zero environmental feedback for hundreds of steps.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.md)
- **PyTorch 2.x Module:** [05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.py)
- **Functional JAX Module:** [05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.py)
- **Native Rust Candle Track:** [05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.rs)
- **Hardware Kernel (CPU / WGSL):** [05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.rs)
- **Golden SafeTensors:** [05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Offline RL Imitation Cql IQL Gail and Decision Transformer](04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 06. Hierarchical RL Options Feudal Her and POMDP Active Inference](06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.ipynb) &rarr;